In [ ]:
# In[1]:

import pandas as pd

# Load CSVs
metric_df = pd.read_csv("metric.csv")
trace_df = pd.read_csv("trace.csv")
log_df = pd.read_csv("log.csv")
error_df = pd.read_csv("error_logs.csv")

# Parse timestamps as UTC datetimes (Unix seconds -> UTC)
for df in (metric_df, trace_df, log_df, error_df):
    df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)

# Helper to build compact one-row summary DataFrame
def make_summary(df, name_col=None):
    row_count = len(df)
    unique_cmdb = df['cmdb_id'].unique()
    unique_cmdb_count = len(unique_cmdb)
    earliest = df['timestamp'].min()
    latest = df['timestamp'].max()
    summary = pd.DataFrame([{
        'rows': row_count,
        'unique_cmdb_count': unique_cmdb_count,
        'unique_cmdb_ids': list(unique_cmdb),
        'earliest_utc': earliest,
        'latest_utc': latest
    }])
    # top name counts if name_col provided
    top_names = None
    if name_col is not None and name_col in df.columns:
        top_names = (df[name_col].value_counts()
                       .head(20)
                       .rename_axis(name_col)
                       .reset_index(name='count'))
    return summary, top_names

# Metric file summary (kpi_name top counts)
metric_summary, metric_top_kpis = make_summary(metric_df, name_col='kpi_name')
metric_sample = metric_df[['timestamp','cmdb_id','kpi_name','value']].head(5)

# Trace file summary (trace_name top counts)
trace_summary, trace_top_traces = make_summary(trace_df, name_col='trace_name')
trace_sample = trace_df[['timestamp','cmdb_id','trace_name','value']].head(5)

# Log file summary (log_name top counts)
log_summary, log_top_logs = make_summary(log_df, name_col='log_name')
log_sample = log_df[['timestamp','cmdb_id','log_name','value']].head(5)

# Error logs summary (top messages)
err_summary, err_top_messages = make_summary(error_df, name_col='message')
err_sample = error_df[['timestamp','cmdb_id','message']].head(5)

# Display compact summaries and top lists and small samples for each file
metric_summary, metric_top_kpis, metric_sample, \
trace_summary, trace_top_traces, trace_sample, \
log_summary, log_top_logs, log_sample, \
err_summary, err_top_messages, err_sample

```
Out[1]:
```
summary = (
    "Telemetry summary (timestamps in UTC):\n"
    "- metric.csv: 1,750 rows, 12 unique cmdb_id values, time range 2024-01-17 10:48:00 to 2024-01-17 11:12:00 UTC. "
    "Top KPI names (by count): cpu, workload, socket, mem (each 275), latency-50 and latency-90 (250), diskio (100), error (50). Sample rows show adservice metrics.\n"
    "- trace.csv: 3,800 rows, 8 unique cmdb_id values, same time range. "
    "Top trace features include trace.from_root.* (duration_mean, duration_p95, error_rate, row_count) and many traces from/to checkoutservice, frontendservice, and productcatalogservice.\n"
    "- log.csv: 460 rows, 10 unique cmdb_id values, same time range. "
    "Top log fields: log.error_count and log.row_count (each 230). Sample log rows show log.error_count = 0 in the displayed samples.\n"
    "- error_logs.csv: empty (0 rows), no error messages recorded.\n\n"
    "Overall: full metric/trace/log coverage exists for the 24-minute window (2024-01-17 10:48–11:12 UTC). Traces show notable activity related to checkoutservice, frontendservice, and productcatalogservice. No raw error log entries are present in error_logs.csv and sampled log.error_count values are zero."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(   rows  unique_cmdb_count                                    unique_cmdb_ids              earliest_utc                latest_utc
0  1750                 12  [adservice, cartservice, checkoutservice, curr... 2024-01-17 10:48:00+00:00 2024-01-17 11:12:00+00:00,      kpi_name  count
0         cpu    275
1    workload    275
2      socket    275
3         mem    275
4  latency-50    250
5  latency-90    250
6      diskio    100
7       error     50,                   timestamp    cmdb_id    kpi_name         value
0 2024-01-17 10:48:00+00:00  adservice         cpu  7.339789e-01
1 2024-01-17 10:48:00+00:00  adservice      diskio  2.496774e+05
2 2024-01-17 10:48:00+00:00  adservice  latency-50  2.992920e-03
3 2024-01-17 10:48:00+00:00  adservice  latency-90  4.609885e-03
4 2024-01-17 10:48:00+00:00  adservice         mem  1.068485e+08,    rows  unique_cmdb_count                                    unique_cmdb_ids              earliest_utc                latest_utc
0  3800                  8  [checkoutservice, currencyservice, emailservic... 2024-01-17 10:48:00+00:00 2024-01-17 11:12:00+00:00,                                       trace_name  count
0                  trace.from_root.duration_mean    175
1                   trace.from_root.duration_p95    175
2                     trace.from_root.error_rate    175
3                      trace.from_root.row_count    175
4           trace.from_checkoutservice.row_count    125
5          trace.from_checkoutservice.error_rate    125
6        trace.from_checkoutservice.duration_p95    125
7       trace.from_checkoutservice.duration_mean    125
8           trace.from_frontendservice.row_count    125
9          trace.from_frontendservice.error_rate    125
10       trace.from_frontendservice.duration_p95    125
11      trace.from_frontendservice.duration_mean    125
12      trace.to_productcatalogservice.row_count    100
13     trace.to_productcatalogservice.error_rate    100
14   trace.to_productcatalogservice.duration_p95    100
15  trace.to_productcatalogservice.duration_mean    100
16        trace.to_checkoutservice.duration_mean     75
17         trace.to_checkoutservice.duration_p95     75
18           trace.to_checkoutservice.error_rate     75
19            trace.to_checkoutservice.row_count     75,                   timestamp          cmdb_id                                trace_name       value
0 2024-01-17 10:48:00+00:00  checkoutservice  trace.from_checkoutservice.duration_mean    0.003552
1 2024-01-17 10:48:00+00:00  checkoutservice   trace.from_checkoutservice.duration_p95    0.005608
2 2024-01-17 10:48:00+00:00  checkoutservice     trace.from_checkoutservice.error_rate    0.000000
3 2024-01-17 10:48:00+00:00  checkoutservice      trace.from_checkoutservice.row_count  203.000000
4 2024-01-17 10:48:00+00:00  checkoutservice  trace.from_frontendservice.duration_mean    0.093603,    rows  unique_cmdb_count                                    unique_cmdb_ids              earliest_utc                latest_utc
0   460                 10  [adservice, cartservice, checkoutservice, curr... 2024-01-17 10:48:00+00:00 2024-01-17 11:12:00+00:00,           log_name  count
0  log.error_count    230
1    log.row_count    230,                   timestamp          cmdb_id         log_name  value
0 2024-01-17 10:48:00+00:00        adservice  log.error_count      0
1 2024-01-17 10:48:00+00:00        adservice    log.row_count    290
2 2024-01-17 10:48:00+00:00      cartservice  log.error_count      0
3 2024-01-17 10:48:00+00:00      cartservice    log.row_count    524
4 2024-01-17 10:48:00+00:00  checkoutservice  log.error_count      0,    rows  unique_cmdb_count unique_cmdb_ids earliest_utc latest_utc
0     0                  0              []          NaT        NaT, Empty DataFrame
Columns: [message, count]
Index: [], Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [])```
```

In [ ]:
# In[2]:

import pandas as pd

# reuse existing metric_df if present, otherwise load
if 'metric_df' not in globals():
    metric_df = pd.read_csv("metric.csv")
# ensure timestamp parsed as UTC datetime
metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)

# target components
components = ['checkoutservice', 'currencyservice', 'emailservice', 'productcatalogservice', 'recommendationservice']

# filter rows for these components
filtered = metric_df[metric_df['cmdb_id'].isin(components)].copy()

# group on full available series for these components and compute count and global p95
agg = (
    filtered
    .groupby(['cmdb_id', 'kpi_name'])['value']
    .agg(count='count', p95=lambda x: x.quantile(0.95))
    .reset_index()
)

# keep output compact: round p95
agg['p95'] = agg['p95'].round(6)

# earliest and latest timestamps for the selected components (UTC)
earliest_ts = filtered['timestamp'].min()
latest_ts = filtered['timestamp'].max()

agg, earliest_ts, latest_ts

```
Out[2]:
```
Summary (plain English):

- You filtered metric.csv for the five services: checkoutservice, currencyservice, emailservice, productcatalogservice, recommendationservice.
- There are 32 distinct (cmdb_id, kpi_name) groups. Each group contains 25 data points (count = 25).
- Time range for these selected components: 2024-01-17 10:48:00 UTC through 2024-01-17 11:12:00 UTC.
- p95 observations (global p95 per group):
  - Memory KPIs (kpi_name = mem) have the largest p95 values (order 1e7–5e7).
  - Socket p95 values are small integers (e.g., 4–9).
  - Latency p95s (latency-50 / latency-90) are in the sub-second range (e.g., latency-90 for checkoutservice ≈ 0.33784 s).
  - CPU p95s are fractional (example: checkoutservice cpu p95 ≈ 0.559335).
  - Workload p95s can be in the tens (example: recommendationservice workload p95 ≈ 19.71891).

If you want the full compact table shown explicitly, I can display all 32 rows.

The original code execution output of IPython Kernel is also provided below for reference:

(                  cmdb_id    kpi_name  count           p95
0         checkoutservice         cpu     25  5.593350e-01
1         checkoutservice  latency-50     25  1.784670e-01
2         checkoutservice  latency-90     25  3.378400e-01
3         checkoutservice         mem     25  1.179118e+07
4         checkoutservice      socket     25  9.000000e+00
..                    ...         ...    ...           ...
27  recommendationservice  latency-50     25  6.030900e-02
28  recommendationservice  latency-90     25  9.209100e-02
29  recommendationservice         mem     25  4.701615e+07
30  recommendationservice      socket     25  4.000000e+00
31  recommendationservice    workload     25  1.971891e+01

[32 rows x 4 columns], Timestamp('2024-01-17 10:48:00+0000', tz='UTC'), Timestamp('2024-01-17 11:12:00+0000', tz='UTC'))

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np

# reuse metric_df if present, otherwise load and parse timestamp as UTC
if 'metric_df' not in globals():
    metric_df = pd.read_csv("metric.csv")
metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)

# target components
components = ['checkoutservice', 'currencyservice', 'emailservice', 'productcatalogservice', 'recommendationservice']

# Rule 9: compute global P95 per (cmdb_id, kpi_name) using the full available series BEFORE filtering to the incident window
p95_df = (
    metric_df[metric_df['cmdb_id'].isin(components)]
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)['value']
    .quantile(0.95)
    .rename(columns={'value': 'p95'})
)

# Incident window (UTC)
start = pd.Timestamp('2024-01-17 10:45:22', tz='UTC')
end = pd.Timestamp('2024-01-17 11:15:22', tz='UTC')

# Filter metric rows to the window for the selected components
window_df = metric_df[
    (metric_df['cmdb_id'].isin(components)) &
    (metric_df['timestamp'] >= start) &
    (metric_df['timestamp'] <= end)
].copy()

# Attach p95 to each row
window_with_p95 = window_df.merge(p95_df, on=['cmdb_id', 'kpi_name'], how='left')

# Detect anomalies where value >= p95
anomalies = window_with_p95[window_with_p95['value'] >= window_with_p95['p95']].copy()

# Aggregate per group for groups that have at least one anomaly
if not anomalies.empty:
    grouped = (
        anomalies
        .groupby(['cmdb_id', 'kpi_name'], as_index=False)
        .agg(
            anomaly_count_in_window=('value', 'size'),
            earliest_anomaly_timestamp_utc=('timestamp', 'min'),
            max_value_in_window=('value', 'max')
        )
    )
    # attach p95 and compute max_over_p95_ratio safely
    grouped = grouped.merge(p95_df, on=['cmdb_id', 'kpi_name'], how='left')
    # handle p95 == 0 to avoid division by zero
    def safe_ratio(row):
        p = row['p95']
        m = row['max_value_in_window']
        if pd.isna(p) or p == 0:
            return np.inf if (not pd.isna(m) and m > 0) else np.nan
        return m / p
    grouped['max_over_p95_ratio'] = grouped.apply(safe_ratio, axis=1)
    # round numeric columns for compactness
    grouped['p95'] = grouped['p95'].round(6)
    grouped['max_value_in_window'] = grouped['max_value_in_window'].round(6)
    grouped['max_over_p95_ratio'] = grouped['max_over_p95_ratio'].round(6)
    # sort by ratio desc and limit to top 20
    anomalies_summary = grouped.sort_values('max_over_p95_ratio', ascending=False).head(20)[
        ['cmdb_id', 'kpi_name', 'p95', 'anomaly_count_in_window', 'earliest_anomaly_timestamp_utc', 'max_value_in_window', 'max_over_p95_ratio']
    ].reset_index(drop=True)
else:
    anomalies_summary = pd.DataFrame(columns=['cmdb_id','kpi_name','p95','anomaly_count_in_window','earliest_anomaly_timestamp_utc','max_value_in_window','max_over_p95_ratio'])

# Top up to 5 anomaly points across all groups, sorted by value/p95 ratio descending
if not anomalies.empty:
    anomalies['p95'] = anomalies['p95'].round(6)
    # compute per-row ratio safely
    anomalies['over_p95_ratio'] = anomalies.apply(lambda r: (np.inf if (r['p95']==0 and r['value']>0) else (r['value']/r['p95']) if r['p95']>0 else np.nan), axis=1)
    anomalies['value'] = anomalies['value'].round(6)
    top_anomaly_points = anomalies.sort_values('over_p95_ratio', ascending=False).head(5)[
        ['timestamp', 'cmdb_id', 'kpi_name', 'value', 'p95']
    ].rename(columns={'timestamp':'timestamp_utc'}).reset_index(drop=True)
else:
    top_anomaly_points = pd.DataFrame(columns=['timestamp_utc','cmdb_id','kpi_name','value','p95'])

# Also return earliest and latest timestamps present for these selected components (UTC) within the original data (not just window)
sel_full = metric_df[metric_df['cmdb_id'].isin(components)]
earliest_ts_sel = sel_full['timestamp'].min() if not sel_full.empty else pd.NaT
latest_ts_sel = sel_full['timestamp'].max() if not sel_full.empty else pd.NaT

# Final concise outputs
anomalies_summary, top_anomaly_points, earliest_ts_sel, latest_ts_sel

```
Out[3]:
```
Summary of anomaly detection (incident window: 2024-01-17 10:45:22 → 2024-01-17 11:15:22 UTC; P95 computed globally from full series):

- Total groups with ≥1 anomaly: 20 (each group had 2 anomaly points in the window).
- Top anomaly by severity (max_value / p95): currencyservice — workload
  - p95 = 81.115282, max_value_in_window = 84.615517, ratio ≈ 1.04315 (earliest anomaly 2024-01-17 10:48:00 UTC).
- Other notable groups near the top: currencyservice latency-90 and latency-50; checkoutservice workload and mem; recommendationservice cpu; productcatalogservice mem and latency; emailservice workload/cpu.

Top up to 5 anomaly points (timestamp UTC, cmdb_id, kpi_name, value, p95):
1. 2024-01-17 10:59:00+00:00 — currencyservice, workload, value=84.615517, p95=81.115282
2. 2024-01-17 10:50:00+00:00 — currencyservice, latency-90, value=0.044608, p95=0.042770
3. 2024-01-17 11:04:00+00:00 — currencyservice, latency-50, value=0.004277, p95=0.004141
4. 2024-01-17 11:08:00+00:00 — checkoutservice, workload, value=1.249850, p95=1.219033
5. 2024-01-17 10:54:00+00:00 — recommendationservice, cpu, value=2.994189, p95=2.922392

Time coverage for the selected components (full metric data): earliest = 2024-01-17 10:48:00 UTC, latest = 2024-01-17 11:12:00 UTC.

If you want, I can display the full 20-row anomalies table produced (sorted by ratio).

The original code execution output of IPython Kernel is also provided below for reference:

(                  cmdb_id    kpi_name           p95  anomaly_count_in_window earliest_anomaly_timestamp_utc  max_value_in_window  max_over_p95_ratio
0         currencyservice    workload  8.111528e+01                        2      2024-01-17 10:48:00+00:00         8.461552e+01            1.043151
1         currencyservice  latency-90  4.277000e-02                        2      2024-01-17 10:50:00+00:00         4.460800e-02            1.042967
2         currencyservice  latency-50  4.141000e-03                        2      2024-01-17 11:04:00+00:00         4.277000e-03            1.032736
3         checkoutservice    workload  1.219033e+00                        2      2024-01-17 10:52:00+00:00         1.249850e+00            1.025280
4   recommendationservice         cpu  2.922392e+00                        2      2024-01-17 10:54:00+00:00         2.994189e+00            1.024568
5         currencyservice         cpu  3.246644e+01                        2      2024-01-17 10:54:00+00:00         3.319065e+01            1.022306
6         checkoutservice         mem  1.179118e+07                        2      2024-01-17 10:57:00+00:00         1.198572e+07            1.016498
7   productcatalogservice  latency-50  3.849000e-03                        2      2024-01-17 11:04:00+00:00         3.908000e-03            1.015456
8            emailservice    workload  1.214413e+00                        2      2024-01-17 10:52:00+00:00         1.232200e+00            1.014646
9         checkoutservice  latency-50  1.784670e-01                        2      2024-01-17 11:02:00+00:00         1.803120e-01            1.010339
10           emailservice         cpu  3.144110e-01                        2      2024-01-17 10:50:00+00:00         3.168200e-01            1.007663
11        checkoutservice         cpu  5.593350e-01                        2      2024-01-17 10:50:00+00:00         5.632770e-01            1.007049
12        checkoutservice  latency-90  3.378400e-01                        2      2024-01-17 11:02:00+00:00         3.398940e-01            1.006080
13  productcatalogservice         mem  1.522194e+07                        2      2024-01-17 11:01:00+00:00         1.530416e+07            1.005401
14  productcatalogservice  latency-90  8.209300e-02                        2      2024-01-17 11:04:00+00:00         8.242300e-02            1.004019
15  recommendationservice    workload  1.971891e+01                        2      2024-01-17 10:49:00+00:00         1.979657e+01            1.003938
16  productcatalogservice    workload  1.504613e+02                        2      2024-01-17 10:49:00+00:00         1.509011e+02            1.002922
17  productcatalogservice         cpu  1.983701e+01                        2      2024-01-17 11:08:00+00:00         1.989189e+01            1.002767
18  recommendationservice  latency-50  6.030900e-02                        2      2024-01-17 11:04:00+00:00         6.041500e-02            1.001755
19        currencyservice         mem  4.519088e+07                        2      2024-01-17 10:50:00+00:00         4.520625e+07            1.000340,               timestamp_utc                cmdb_id    kpi_name      value        p95
0 2024-01-17 10:59:00+00:00        currencyservice    workload  84.615517  81.115282
1 2024-01-17 10:50:00+00:00        currencyservice  latency-90   0.044608   0.042770
2 2024-01-17 11:04:00+00:00        currencyservice  latency-50   0.004277   0.004141
3 2024-01-17 11:08:00+00:00        checkoutservice    workload   1.249850   1.219033
4 2024-01-17 10:54:00+00:00  recommendationservice         cpu   2.994189   2.922392, Timestamp('2024-01-17 10:48:00+0000', tz='UTC'), Timestamp('2024-01-17 11:12:00+0000', tz='UTC'))```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np

# reuse trace_df if present, otherwise load and parse timestamp as UTC
if 'trace_df' not in globals():
    trace_df = pd.read_csv("trace.csv")
trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)

# target components
components = ['currencyservice', 'checkoutservice', 'productcatalogservice', 'recommendationservice', 'emailservice']

# Rule 9: compute global P95 per (cmdb_id, trace_name) using the full available series BEFORE window filtering
p95_df = (
    trace_df[trace_df['cmdb_id'].isin(components)]
    .groupby(['cmdb_id', 'trace_name'], as_index=False)['value']
    .quantile(0.95)
    .rename(columns={'value': 'p95'})
)

# Incident window (UTC)
start = pd.Timestamp('2024-01-17 10:45:22', tz='UTC')
end = pd.Timestamp('2024-01-17 11:15:22', tz='UTC')

# Filter trace rows to the window for selected components
window_df = trace_df[
    (trace_df['cmdb_id'].isin(components)) &
    (trace_df['timestamp'] >= start) &
    (trace_df['timestamp'] <= end)
].copy()

# Attach p95 to each row
window_with_p95 = window_df.merge(p95_df, on=['cmdb_id', 'trace_name'], how='left')

# Detect anomalies where value >= p95
anomalies = window_with_p95[window_with_p95['value'] >= window_with_p95['p95']].copy()

# Aggregate per group for groups that have at least one anomaly
if not anomalies.empty:
    grouped = (
        anomalies
        .groupby(['cmdb_id', 'trace_name'], as_index=False)
        .agg(
            anomaly_count_in_window=('value', 'size'),
            earliest_anomaly_timestamp_utc=('timestamp', 'min'),
            max_value_in_window=('value', 'max')
        )
    )
    grouped = grouped.merge(p95_df, on=['cmdb_id', 'trace_name'], how='left')
    # safe ratio computation
    def safe_ratio(row):
        p = row['p95']
        m = row['max_value_in_window']
        if pd.isna(p) or p == 0:
            return np.inf if (not pd.isna(m) and m > 0) else np.nan
        return m / p
    grouped['max_over_p95_ratio'] = grouped.apply(safe_ratio, axis=1)
    # rounding for compactness
    grouped['p95'] = grouped['p95'].round(6)
    grouped['max_value_in_window'] = grouped['max_value_in_window'].round(6)
    grouped['max_over_p95_ratio'] = grouped['max_over_p95_ratio'].round(6)
    anomalies_summary = (
        grouped
        .sort_values('max_over_p95_ratio', ascending=False)
        .head(20)[
            ['cmdb_id', 'trace_name', 'p95', 'anomaly_count_in_window',
             'earliest_anomaly_timestamp_utc', 'max_value_in_window', 'max_over_p95_ratio']
        ]
        .reset_index(drop=True)
    )
    # top up to 5 anomaly points across these groups
    anomalies['p95'] = anomalies['p95'].round(6)
    anomalies['value'] = anomalies['value'].round(6)
    anomalies['over_p95_ratio'] = anomalies.apply(
        lambda r: (np.inf if (r['p95']==0 and r['value']>0) else (r['value']/r['p95']) if r['p95']>0 else np.nan),
        axis=1
    )
    top_anomaly_points = (
        anomalies
        .sort_values('over_p95_ratio', ascending=False)
        .head(5)[['timestamp', 'cmdb_id', 'trace_name', 'value', 'p95']]
        .rename(columns={'timestamp':'timestamp_utc'})
        .reset_index(drop=True)
    )
else:
    anomalies_summary = pd.DataFrame(columns=[
        'cmdb_id', 'trace_name', 'p95', 'anomaly_count_in_window',
        'earliest_anomaly_timestamp_utc', 'max_value_in_window', 'max_over_p95_ratio'
    ])
    top_anomaly_points = pd.DataFrame(columns=['timestamp_utc','cmdb_id','trace_name','value','p95'])

# Return concise results
anomalies_summary, top_anomaly_points

```
Out[4]:
```
Summary of trace-based anomaly detection (incident window 2024-01-17 10:45:22 → 2024-01-17 11:15:22 UTC; P95 computed globally from full series):

- Groups with ≥1 anomaly: 20 groups (each reported 2 anomaly points in the window).
- Most severe anomaly (by max_value / p95): currencyservice — trace.from_root.duration_p95
  - p95 = 0.000243, max value = 0.000838, ratio ≈ 3.4405 (earliest anomaly 2024-01-17 11:06:00 UTC).
- Other high-ranking anomalies (by ratio): 
  - currencyservice — trace.from_root.duration_mean (ratio ≈ 2.1679)
  - checkoutservice — trace.to_currencyservice.duration_p95 (ratio ≈ 2.1107)
  - currencyservice — trace.from_checkoutservice.duration_p95 (ratio ≈ 2.1107)
  - recommendationservice / productcatalogservice traces showing elevated duration means (ratios ≈ 1.65)
  - several checkoutservice traces and emailservice traces with smaller but >1 ratios (≈1.34 → 1.10).

Top up to 5 sample anomaly points (timestamp UTC, cmdb_id, trace_name, value, p95):
1. 2024-01-17 11:06:00+00:00 — currencyservice, trace.from_root.duration_p95, value=0.000838, p95=0.000243
2. 2024-01-17 11:06:00+00:00 — currencyservice, trace.from_root.duration_mean, value=0.000199, p95=0.000092
3. 2024-01-17 10:51:00+00:00 — checkoutservice, trace.to_currencyservice.duration_p95, value=0.000708, p95=0.000335
4. 2024-01-17 10:51:00+00:00 — currencyservice, trace.from_checkoutservice.duration_p95, value=0.000708, p95=0.000335
5. 2024-01-17 11:04:00+00:00 — productcatalogservice, trace.from_recommendationservice.duration_mean, value=0.000508, p95=0.000307

Interpretation: currencyservice shows the largest trace-duration spikes (notably trace.from_root.*), and there are cross-service duration increases involving checkoutservice and productcatalogservice as well. If you want, I can display the full 20-row anomalies table.

The original code execution output of IPython Kernel is also provided below for reference:

(                  cmdb_id                                      trace_name       p95  anomaly_count_in_window earliest_anomaly_timestamp_utc  max_value_in_window  max_over_p95_ratio
0         currencyservice                    trace.from_root.duration_p95  0.000243                        2      2024-01-17 11:06:00+00:00             0.000838            3.440542
1         currencyservice                   trace.from_root.duration_mean  0.000092                        2      2024-01-17 11:06:00+00:00             0.000199            2.167921
2         checkoutservice           trace.to_currencyservice.duration_p95  0.000335                        2      2024-01-17 10:51:00+00:00             0.000708            2.110661
3         currencyservice         trace.from_checkoutservice.duration_p95  0.000335                        2      2024-01-17 10:51:00+00:00             0.000708            2.110661
4   recommendationservice    trace.to_productcatalogservice.duration_mean  0.000307                        2      2024-01-17 11:04:00+00:00             0.000508            1.653554
5   productcatalogservice  trace.from_recommendationservice.duration_mean  0.000307                        2      2024-01-17 11:04:00+00:00             0.000508            1.653554
6   productcatalogservice        trace.from_frontendservice.duration_mean  0.000300                        2      2024-01-17 11:09:00+00:00             0.000421            1.403250
7         checkoutservice         trace.from_frontendservice.duration_p95  0.461418                        2      2024-01-17 10:59:00+00:00             0.619938            1.343549
8   recommendationservice                    trace.from_root.duration_p95  0.000154                        2      2024-01-17 10:57:00+00:00             0.000191            1.241862
9            emailservice                   trace.from_root.duration_mean  0.000054                        2      2024-01-17 10:54:00+00:00             0.000065            1.202337
10        checkoutservice            trace.to_paymentservice.duration_p95  0.000736                        2      2024-01-17 11:00:00+00:00             0.000883            1.200778
11        checkoutservice        trace.from_checkoutservice.duration_mean  0.017597                        2      2024-01-17 11:03:00+00:00             0.020888            1.187066
12        checkoutservice          trace.to_checkoutservice.duration_mean  0.017597                        2      2024-01-17 11:03:00+00:00             0.020888            1.187066
13        checkoutservice           trace.to_paymentservice.duration_mean  0.000331                        2      2024-01-17 11:06:00+00:00             0.000388            1.171872
14           emailservice         trace.from_checkoutservice.duration_p95  0.000623                        2      2024-01-17 10:48:00+00:00             0.000709            1.137743
15        checkoutservice              trace.to_emailservice.duration_p95  0.000623                        2      2024-01-17 10:48:00+00:00             0.000709            1.137743
16           emailservice                    trace.from_root.duration_p95  0.000107                        2      2024-01-17 10:54:00+00:00             0.000118            1.101987
17  productcatalogservice                   trace.from_root.duration_mean  0.036376                        2      2024-01-17 11:08:00+00:00             0.039862            1.095832
18        currencyservice        trace.from_frontendservice.duration_mean  0.000474                        2      2024-01-17 10:56:00+00:00             0.000519            1.094489
19        checkoutservice          trace.to_currencyservice.duration_mean  0.001894                        2      2024-01-17 10:50:00+00:00             0.002057            1.086220,               timestamp_utc                cmdb_id                                      trace_name     value       p95
0 2024-01-17 11:06:00+00:00        currencyservice                    trace.from_root.duration_p95  0.000838  0.000243
1 2024-01-17 11:06:00+00:00        currencyservice                   trace.from_root.duration_mean  0.000199  0.000092
2 2024-01-17 10:51:00+00:00        checkoutservice           trace.to_currencyservice.duration_p95  0.000708  0.000335
3 2024-01-17 10:51:00+00:00        currencyservice         trace.from_checkoutservice.duration_p95  0.000708  0.000335
4 2024-01-17 11:04:00+00:00  productcatalogservice  trace.from_recommendationservice.duration_mean  0.000508  0.000307)```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np

# reuse log_df and error_df if present, otherwise load and parse timestamps as UTC
if 'log_df' not in globals():
    log_df = pd.read_csv("log.csv")
if 'error_df' not in globals():
    # file is named error_logs.csv per task description
    error_df = pd.read_csv("error_logs.csv")

# parse timestamps as UTC
log_df['timestamp'] = pd.to_datetime(log_df['timestamp'], unit='s', utc=True)
error_df['timestamp'] = pd.to_datetime(error_df['timestamp'], unit='s', utc=True)

# target components
components = ['checkoutservice', 'currencyservice', 'emailservice', 'productcatalogservice', 'recommendationservice']

# Rule 9: compute global P95 and P05 per (cmdb_id, log_name) using the full available series BEFORE window filtering
log_sel_full = log_df[log_df['cmdb_id'].isin(components)].copy()

if not log_sel_full.empty:
    p95 = log_sel_full.groupby(['cmdb_id', 'log_name'], as_index=False)['value'].quantile(0.95).rename(columns={'value':'p95'})
    p05 = log_sel_full.groupby(['cmdb_id', 'log_name'], as_index=False)['value'].quantile(0.05).rename(columns={'value':'p05'})
    thresholds = p95.merge(p05, on=['cmdb_id','log_name'], how='outer')
else:
    thresholds = pd.DataFrame(columns=['cmdb_id','log_name','p95','p05'])

# Incident window (UTC)
start = pd.Timestamp('2024-01-17 10:45:22', tz='UTC')
end = pd.Timestamp('2024-01-17 11:15:22', tz='UTC')

# Filter log rows to the window for selected components
window_df = log_sel_full[
    (log_sel_full['timestamp'] >= start) &
    (log_sel_full['timestamp'] <= end)
].copy()

# Attach thresholds to each row
window_with_thresh = window_df.merge(thresholds, on=['cmdb_id','log_name'], how='left')

# Detect anomalies: high if value >= p95; low if value <= p05
def detect_direction(row):
    p95v = row['p95']
    p05v = row['p05']
    val = row['value']
    high = (pd.notna(p95v)) and (val >= p95v)
    low = (pd.notna(p05v)) and (val <= p05v)
    if high:
        return 'high'
    if low:
        return 'low'
    return None

window_with_thresh['direction'] = window_with_thresh.apply(detect_direction, axis=1)
anomalies = window_with_thresh[window_with_thresh['direction'].notna()].copy()

# Aggregate per (cmdb_id, log_name, direction)
if not anomalies.empty:
    # For 'high' extreme_value is max; for 'low' extreme_value is min
    def agg_extreme(df):
        direction = df.name[2]  # groupby keys: (cmdb_id, log_name, direction)
        if direction == 'high':
            extreme = df['value'].max()
        else:
            extreme = df['value'].min()
        return pd.Series({
            'anomaly_count_in_window': df.shape[0],
            'earliest_anomaly_timestamp_utc': df['timestamp'].min(),
            'extreme_value_in_window': extreme
        })

    grouped = (
        anomalies
        .groupby(['cmdb_id','log_name','direction'])
        .apply(agg_extreme)
        .reset_index()
    )

    # attach thresholds
    grouped = grouped.merge(thresholds, on=['cmdb_id','log_name'], how='left')

    # compute severity_metric
    def compute_severity(row):
        dirc = row['direction']
        extreme = row['extreme_value_in_window']
        p95v = row['p95']
        p05v = row['p05']
        if dirc == 'high':
            if pd.isna(p95v) or p95v == 0:
                return np.inf if (not pd.isna(extreme) and extreme > 0) else np.nan
            return extreme / p95v
        else:  # low
            if pd.isna(p05v) or p05v == 0:
                return np.inf if (not pd.isna(extreme) and extreme > 0) else np.nan
            return extreme / p05v

    grouped['severity_metric'] = grouped.apply(compute_severity, axis=1)

    # Round numeric columns for compact display
    grouped['p95'] = grouped['p95'].round(6)
    grouped['p05'] = grouped['p05'].round(6)
    grouped['extreme_value_in_window'] = grouped['extreme_value_in_window'].round(6)
    grouped['severity_metric'] = grouped['severity_metric'].round(6)

    # Select columns and sort by severity descending, limit top 20
    anomalies_summary = (
        grouped[['cmdb_id','log_name','p95','p05','anomaly_count_in_window',
                 'earliest_anomaly_timestamp_utc','extreme_value_in_window','direction','severity_metric']]
        .sort_values('severity_metric', ascending=False)
        .head(20)
        .reset_index(drop=True)
    )
else:
    anomalies_summary = pd.DataFrame(columns=['cmdb_id','log_name','p95','p05','anomaly_count_in_window',
                                              'earliest_anomaly_timestamp_utc','extreme_value_in_window','direction','severity_metric'])

# Up to 5 sample anomaly rows with requested columns
if not anomalies.empty:
    # compute per-row severity for sorting samples (value relative to its p95/p05)
    def row_severity(r):
        if r['direction']=='high':
            if pd.isna(r['p95']) or r['p95']==0:
                return np.inf if r['value']>0 else np.nan
            return r['value']/r['p95']
        else:
            if pd.isna(r['p05']) or r['p05']==0:
                return np.inf if r['value']>0 else np.nan
            return r['value']/r['p05']
    anomalies['row_sev'] = anomalies.apply(row_severity, axis=1)
    anomalies['value'] = anomalies['value'].round(6)
    anomalies['p95'] = anomalies['p95'].round(6)
    anomalies['p05'] = anomalies['p05'].round(6)
    sample_anomalies = (
        anomalies.sort_values('row_sev', ascending=False)
        .head(5)[['timestamp','cmdb_id','log_name','value','p95','p05','direction']]
        .rename(columns={'timestamp':'timestamp_utc'})
        .reset_index(drop=True)
    )
else:
    sample_anomalies = pd.DataFrame(columns=['timestamp_utc','cmdb_id','log_name','value','p95','p05','direction'])

# Search error_logs.csv for any rows with cmdb_id in components within the incident window and return up to 10 matches
err_matches = error_df[
    (error_df['cmdb_id'].isin(components)) &
    (error_df['timestamp'] >= start) &
    (error_df['timestamp'] <= end)
].copy()

if not err_matches.empty:
    err_matches = err_matches[['timestamp','cmdb_id','message']].rename(columns={'timestamp':'timestamp_utc'}).sort_values('timestamp_utc').head(10).reset_index(drop=True)
else:
    err_matches = pd.DataFrame(columns=['timestamp_utc','cmdb_id','message'])

# Final concise outputs
anomalies_summary, sample_anomalies, err_matches

```
Out[5]:
```
Summary (plain English):

- Incident window: 2024-01-17 10:45:22 → 2024-01-17 11:15:22 UTC.

- Primary log anomalies are in log.row_count for multiple services:
  - recommendationservice: high spike to 610 (p95=582.6, severity ≈ 1.047), earliest high at 2024-01-17 10:49:00 UTC; also a low at 212 (p05=405.0, severity ≈ 0.523) at 10:48:00.
  - currencyservice: high spikes to 2560 (p95=2464.8, severity ≈ 1.039), earliest high at 10:53:00 UTC; also low values observed (~889) earlier (severity ≈ 0.517) at 10:48:00.
  - emailservice: high to 36 (p95=34.8, severity ≈ 1.034), earliest high at 10:52:00 UTC; also low values (~10) around 10:57:00.
  - checkoutservice: high to 108 (p95=104.4, severity ≈ 1.034), earliest high at 10:52:00 UTC; also low values (~30) at 10:57:00.
  - product/recommendation/checkout row_count anomalies appear both as spikes and drops (each group reported anomaly_count_in_window = 2 or 3 in the window).

- log.error_count for these services is all zeros (p95 = p05 = 0.0). These rows were flagged as anomalies by the >=p95 rule, but severity is undefined (division-by-zero), which in practice indicates there were no non-zero error counts recorded in the baseline or window.

- Sample anomaly points (from the window):
  - 2024-01-17 10:53:00 — recommendationservice, log.row_count = 610 (p95=582.6) — high
  - 2024-01-17 10:59:00 — currencyservice, log.row_count = 2560 (p95=2464.8) — high
  - 2024-01-17 10:52:00 — emailservice, log.row_count = 36 (p95=34.8) — high
  - 2024-01-17 10:52:00 — checkoutservice, log.row_count = 108 (p95=104.4) — high
  - 2024-01-17 10:53:00 — currencyservice, log.row_count = 2479 (p95=2464.8) — high

- error_logs.csv search: no matching error log rows for these services in the incident window (no error messages returned).

Interpretation: the most notable log-level signals are moderate spikes (and some drops) in log.row_count across recommendationservice, currencyservice, emailservice, and checkoutservice during the window. There are no non-zero log.error_count or error_log entries for these services in the same window. If you want, I can display the full anomalies table or filter to a particular service for more detail.

The original code execution output of IPython Kernel is also provided below for reference:

(                  cmdb_id         log_name     p95     p05  anomaly_count_in_window earliest_anomaly_timestamp_utc  extreme_value_in_window direction  severity_metric
0   recommendationservice    log.row_count   582.6   405.0                        2      2024-01-17 10:49:00+00:00                      610      high         1.047031
1         currencyservice    log.row_count  2464.8  1720.0                        2      2024-01-17 10:53:00+00:00                     2560      high         1.038624
2            emailservice    log.row_count    34.8    21.0                        2      2024-01-17 10:52:00+00:00                       36      high         1.034483
3         checkoutservice    log.row_count   104.4    63.0                        2      2024-01-17 10:52:00+00:00                      108      high         1.034483
4   recommendationservice    log.row_count   582.6   405.0                        2      2024-01-17 10:48:00+00:00                      212       low         0.523457
5         currencyservice    log.row_count  2464.8  1720.0                        2      2024-01-17 10:48:00+00:00                      889       low         0.516860
6         checkoutservice    log.row_count   104.4    63.0                        3      2024-01-17 10:57:00+00:00                       30       low         0.476190
7            emailservice    log.row_count    34.8    21.0                        3      2024-01-17 10:57:00+00:00                       10       low         0.476190
8         checkoutservice  log.error_count     0.0     0.0                       25      2024-01-17 10:48:00+00:00                        0      high              NaN
9         currencyservice  log.error_count     0.0     0.0                       25      2024-01-17 10:48:00+00:00                        0      high              NaN
10           emailservice  log.error_count     0.0     0.0                       25      2024-01-17 10:48:00+00:00                        0      high              NaN
11  recommendationservice  log.error_count     0.0     0.0                       25      2024-01-17 10:48:00+00:00                        0      high              NaN,               timestamp_utc                cmdb_id       log_name  value     p95     p05 direction
0 2024-01-17 10:53:00+00:00  recommendationservice  log.row_count    610   582.6   405.0      high
1 2024-01-17 10:59:00+00:00        currencyservice  log.row_count   2560  2464.8  1720.0      high
2 2024-01-17 10:52:00+00:00           emailservice  log.row_count     36    34.8    21.0      high
3 2024-01-17 10:52:00+00:00        checkoutservice  log.row_count    108   104.4    63.0      high
4 2024-01-17 10:53:00+00:00        currencyservice  log.row_count   2479  2464.8  1720.0      high, Empty DataFrame
Columns: [timestamp_utc, cmdb_id, message]
Index: [])```
```